# E05: Decoradores y Closures

**Nivel:** Intermedio
**Objetivo:** Dominar dos de los conceptos más elegantes de Python: las *closures* (cápsulas de memoria) y los *decoradores* (envolturas de regalo). Aprenderás cómo se construyen por dentro, por qué `functools.wraps` es esencial, y cómo explotar los decoradores integrados del estándar.

> Este es el tema que separa a quienes "escriben Python" de quienes "piensan en Python".

## Objetivos

Al terminar este notebook podrás:

1. **Entender** las funciones como ciudadanos de primera clase (asignarlas, pasarlas y retornarlas).
2. **Construir** closures y explicar por qué se forman mediante `nonlocal` y el *enclosing scope*.
3. **Crear** tus propios decoradores básicos y con argumentos, preservando metadata con `functools.wraps`.
4. **Aplicar** decoradores de la stdlib como `@lru_cache`, `@cache`, `@property`, `@classmethod` y `@staticmethod`.
5. **Componer** múltiples decoradores y entender el orden en el que se aplican (stack).

## Analogía

### Decoradores = "envolturas de regalo" 🎁

Imagina que tienes un regalo (una función). Puedes envolverlo en un papel precioso (un decorador) **sin modificar el contenido del regalo**. Quien lo recibe abre la envoltura y disfruta del regalo, pero la envoltura añade valor: un lazo, una tarjeta, protección. Lo mismo hace un decorador: envuelve una función y le añade comportamiento (timing, validación, caché) **sin tocar su código original**.

```
         regalo                          regalo envuelto
   ┌───────────────┐                   ┌───────────────┐
   │   funcion()   │    ------>        │  envoltura()  │
   │ contenido rico│    decorador      │  ┌─────────┐  │
   └───────────────┘                   │  │funcion()│  │
                                       │  └─────────┘  │
                                       └───────────────┘
```

### Closures = "cápsulas de memoria" 🧠

Una closure es como una cápsula que **captura y recuerda** el contexto en el que nació. Aunque la función original (el *enclosing scope*) ya terminó, la closure conserva el acceso a esas variables como si las llevara en una mochila interna. Es memoria portátil: la función "recuerda" de dónde vino.

## 1. Funciones de primera clase

En Python las funciones son **objetos de primera clase** (first-class citizens): puedes asignarlas a variables, pasarlas como argumentos a otras funciones, y retornarlas desde otras funciones. Esto es el cimiento sobre el que se construyen los decoradores.

### 1.1 Asignar una función a una variable

Una función es un objeto como cualquier otro. Al asignarla **no usamos paréntesis** (esos ejecutan); asignamos la referencia.

In [ ]:
def saludar(nombre: str) -> str:
    return f"Hola, {nombre}!"

# Asignamos la REFERENCIA (no la llamamos, no hay paréntesis)
mi_funcion = saludar

print(mi_funcion("Ana"))
print("Son el mismo objeto?", mi_funcion is saludar)
print("Nombre:", saludar.__name__)

### 1.2 Pasar una función como argumento

Una función puede recibir otra función como parámetro y ejecutarla *cuando quiera*. Aquí `aplicar_a_cada_uno` recibe una función `transformacion` y la aplica a cada elemento.

In [ ]:
def aplicar(fn, datos: list) -> list:
    """Aplica la función fn a cada elemento de datos."""
    return [fn(x) for x in datos]

def doble(x: int) -> int:
    return x * 2

print(aplicar(doble, [1, 2, 3, 4]))  # [2, 4, 6, 8]
print(aplicar(str.upper, ["hola", "mundo"]))  # ['HOLA', 'MUNDO']

### 1.3 Retornar una función desde otra función

Una función puede *fabricar* y devolver otra función. Esto es la semilla de los decoradores: la función externa prepara un contexto y la interna (retornada) lo usa.

In [ ]:
def crear_multiplicador(factor: int):
    """Devuelve una función que multiplica por 'factor'."""
    def multiplicar(x: int) -> int:
        return x * factor
    return multiplicar

por_tres = crear_multiplicador(3)
por_diez = crear_multiplicador(10)

print(por_tres(5))   # 15
print(por_diez(5))   # 50
print(por_tres.__name__)  # multiplicar

### 1.4 `__name__` y el aviso de los decoradores

Cuando envuelves una función, la función que realmente se expone es la interna. Sin cuidado, **perderías** el nombre y docstring originales. Mira el efecto antes de aplicar decoradores:

In [ ]:
def envolver(fn):
    def interior():
        print("--> entrando")
        return fn()
    return interior

@envolver
def calcular():
    """Hago un cálculo importante."""
    return 42

print(calcular())
print("Nombre:", calcular.__name__)   # interior (¡perdido!)
print("Docstring:", calcular.__doc__)  # None (¡perdido!)

> **Atención:** el `__name__` cambió a `interior`. Esto rompe herramientas de depuración, testing y documentación automática. Veremos más adelante cómo `functools.wraps` lo arregla.

## 2. Closures

Una **closure** es una función interna que *captura* las variables del ámbito (*scope*) en el que fue definida, incluso después de que ese ámbito haya terminado. Esa "mochila de memoria" se guarda en el atributo `__closure__`.

### 2.1 ¿Qué es y por qué se forma?

En `crear_multiplicador`, la función interna `multiplicar` referencia `factor` (que pertenece al *enclosing scope* de `crear_multiplicador`). Cuando retornamos `multiplicar`, Python conserva `factor` en una *célula* (cell) porque aún se usa. Sin esa referencia a una variable externa, **no habría closure**.

In [ ]:
def crear_contador():
    """Closure: recuerda 'cuenta' entre llamadas."""
    cuenta = 0  # variable del enclosing scope

    def incrementar() -> int:
        nonlocal cuenta  # declaremos que modificaremos la externa
        cuenta += 1
        return cuenta

    return incrementar

contador = crear_contador()

print(contador())  # 1
print(contador())  # 2
print(contador())  # 3
print("Tiene closure?", contador.__closure__ is not None)

### 2.2 El papel de `nonlocal`

Si solo **leemos** una variable externa, Python no la marcará como local en la interna. Pero si queremos **modificarla** (como `cuenta += 1`), Python la trataría como local de la interna y fallaría con `UnboundLocalError`. La palabra clave `nonlocal` le dice al intérprete: "esta variable pertenece a un ámbito encerrante; modifícala ahí".

In [ ]:
def sin_nonlocal():
    x = 0
    def interna():
        x += 1   # ¡Error! Python piensa que x es local aquí
        return x
    return interna

try:
    f = sin_nonlocal()
    f()
except UnboundLocalError as e:
    print("Error como se esperaba:", e)

### 2.3 Inspección con `__closure__`

El atributo `__closure__` es una tupla de *células*. Cada `cell` tiene un `cell_contents` con el valor capturado. Basta con **leer** una externa para que aparezca.

In [ ]:
def crear_prefijo(pfx: str):
    def con_prefijo(texto: str) -> str:
        return f"{pfx}{texto}"
    return con_prefijo

greet_de = crear_prefijo("Hallo, ")

print(greet_de("Welt"))
print("¿Es closure?", greet_de.__closure__ is not None)
print("Valores capturados:", [c.cell_contents for c in greet_de.__closure__])
print("Nombres? -> Ver código:", greet_de.__code__.co_freevars)

### 2.4 Comparación: con y sin closure

Si una función interna **no referencia ninguna variable externa**, Python no crea una closure (optimización): `__closure__` será `None`.

In [ ]:
def fabricar(total_externo):
    def usa_externo():
        return total_externo            # usa la externa -> closure
    def no_usa_externo():
        return 100                     # no la usa -> sin closure
    return usa_externo, no_usa_externo

usa, no_usa = fabricar(50)
print("Con closure:", usa.__closure__ is not None)      # True
print("Sin closure:", no_usa.__closure__ is not None)   # False

## 3. Decoradores básicos

Un **decorador** es simplemente una *función (o clase) que recibe una función y devuelve otra función* (normalmente una envoltura). La sintaxis con `@` es azúcar sintáctico: es equivalente a reasignar el nombre.

```python
@decorador
def f():
    ...

# es equivalente a:
f = decorador(f)
```

### 3.1 Un decorador que envuelve

Creamos un decorador que imprime cuándo se llama la función, sin tocar el código de esta.

In [ ]:
import time

def tiempo(fn):
    """DeCorador: mide y muestra el tiempo de ejecución."""
    def envoltura(*args, **kwargs):
        inicio = time.perf_counter()
        resultado = fn(*args, **kwargs)
        fin = time.perf_counter()
        print(f"{fn.__name__} tardó {fin - inicio:.6f} s")
        return resultado
    return envoltura

@tiempo
def sumar_numeros(n: int) -> int:
    return sum(range(n))

print("Resultado:", sumar_numeros(1_000_000))

> **`*args, **kwargs`**: la envoltura acepta cualquier argumento posicional y por clave, así el decorador funciona con *cualquier* función.

### 3.2 La sintaxis `@` por dentro

Veamos la equivalencia exacta: `@decorador` es `f = decorador(f)`. En el ejemplo anterior, `sumar_numeros` ya no es la función original, sino `envoltura`.

In [ ]:
def mayusculas(fn):
    def envoltura(*args, **kwargs):
        r = fn(*args, **kwargs)
        return r.upper()
    return envoltura

# Sin azúcar sintáctico:
def saludo_triste():
    return "hola mundo"

saludo_triste = mayusculas(saludo_triste)
print(saludo_triste())

# Equivalente pero con @ :
@mayusculas
def saludo_alegre():
    return "hola mundo"

print(saludo_alegre())

### 3.3 `functools.wraps`: por qué es esencial

Como vimos en la sección 1.4, la envoltura se hace pasar por la función original, **perdiendo su identidad** (`__name__`, `__doc__`, firmas...). `functools.wraps` copia esa metadata desde la función original hacia la envoltura. Sin él, los `help()`, los logs y los tests se rompen.

In [ ]:
import functools

def tiempo(fn):
    @functools.wraps(fn)   # copia __name__, __doc__, __module__, ...
    def envoltura(*args, **kwargs):
        inicio = time.perf_counter()
        r = fn(*args, **kwargs)
        print(f"{fn.__name__} tardó {time.perf_counter() - inicio:.4f} s")
        return r
    return envoltura

@tiempo
def calcular_total(items):
    """Suma todos los items."""
    return sum(items)

print(calcular_total([1, 2, 3]))
print("Nombre preservado:", calcular_total.__name__)   # calcular_total
print("Docstring preservado:", calcular_total.__doc__)
print("Original oculto en __wrapped__:", calcular_total.__wrapped__)

> **Regla de oro:** **siempre** usa `@functools.wraps` en tus envolturas. Nunca escribas un decorador sin él (salvo que tengas una razón muy concreta).

## 4. Decoradores con argumentos

¿Y si el decorador necesita sus propios parámetros, como `@reintentar(intentos=3)`? Ya no basta con una doble función: necesitamos **tres niveles** (patrón de triple función):

```
decorador_con_argumentos(arg)        -> devuelve el decorador real
    -> decorador(fn)                  -> devuelve la envoltura
        -> envoltura(*args, **kwargs) -> ejecuta
```

In [ ]:
import functools

def repetir(veces: int = 2):
    """Factory: devuelve un decorador que repite la llamada 'veces' veces."""
    def decorador(fn):
        @functools.wraps(fn)
        def envoltura(*args, **kwargs):
            for _ in range(veces):
                fn(*args, **kwargs)
        return envoltura
    return decorador

@repetir(veces=3)
def avisar(msg: str):
    print("AVISO:", msg)

avisar("mantenimiento programado")
print("--")
# Se puede usar SIN paréntesis si quieres el valor por defecto:
@repetir()
def avisar2(msg: str):
    print("nota:", msg)

avisar2("hola")

### 4.1 ¿Por qué existe el patrón de triple función?

Sin argumentos, `@decorador` pasa la *función* directamente: `f = decorador(f)`. Con argumentos, `@decorador(2)` **primero** evalúa `decorador(2)` (una llamada normal que devuelve el decorador real) y **después** aplica ese resultado a `f`: `f = decorador(2)(f)`. Por eso la capa externa (el *factory*) maneja los argumentos.

In [ ]:
# Demostración de las dos formas de aplicación
import functools

def duplicador(fn):
    @functools.wraps(fn)
    def e(*args, **kwargs):
        return fn(*args, **kwargs) * 2
    return e

# Forma 1: aplicar directo
def uno():
    return 10
uno = duplicador(uno)

# Forma 2 (con arandela): decorador(fn) -> e
def arandela(extra=0):
    def deco(fn):
        @functools.wraps(fn)
        def e2():
            return fn() + extra
        return e2
    return deco

print("Directo:", uno())

def dos():
    return 5
dos = arandela(extra=100)(dos)
print("Con arandela:", dos())

## 5. Decoradores de clase

Los decoradores **no solo decoran funciones**: también pueden decorar clases. Reciben la clase y devuelven una clase modificada (o una envoltura). Es una técnica de *metaprogramación* para registrar, validar o enriquecer clases.

In [ ]:
# Decorador que añade un método a la clase
def con_repr(fn):
    """Decorador de CLASE: añade un __repr__ automático."""
    def __repr__(self):
        campos = ", ".join(f"{k}={v!r}" for k, v in vars(self).items())
        return f"{self.__class__.__name__}({campos})"
    fn.__repr__ = __repr__
    return fn

@con_repr
class Producto:
    def __init__(self, nombre: str, precio: float):
        self.nombre = nombre
        self.precio = precio

p = Producto("laptop", 999.99)
print(p)
print(repr(p))

In [ ]:
# Decorador que REGISTRA la clase en un catálogo global
CATALOGO = {}

def registrar(mi_clase):
    CATALOGO[mi_clase.__name__] = mi_clase
    return mi_clase

@registrar
class Animal:
    pass

@registrar
class Planta:
    pass

print("Clases registradas:", sorted(CATALOGO))
print("Instanciar desde catálogo:", CATALOGO["Animal"]())

> Ejemplos reales de decoradores de clase: `@dataclass`, `@frozen`, `@total_ordering`. Décora la clase y en `__init_subclass__`/metaclasses hacen su magia.

## 6. Memoización con `@lru_cache` / `@cache`

La **memoización** guarda los resultados ya calculados para no repetirlos. Es un caso de uso *perfecto* para decoradores, y la stdlib lo trae listo con `functools.lru_cache` y `functools.cache`.

### 6.1 Fibonacci: el ejemplo clásico

Sin caché, `fib` recursivo explota exponencialmente ($O(2^n)$). Con `@cache`, cada valor se calcula una sola vez ($O(n)$).

In [ ]:
from functools import cache, lru_cache

def fib_naive(n: int) -> int:
    if n < 2:
        return n
    return fib_naive(n - 1) + fib_naive(n - 2)

@cache
def fib_cache(n: int) -> int:
    if n < 2:
        return n
    return fib_cache(n - 1) + fib_cache(n - 2)

import time

t0 = time.perf_counter(); fib_naive(30); t1 = time.perf_counter()
t2 = time.perf_counter(); fib_cache(30); t3 = time.perf_counter()

print(f"fib_naive(30): {t1-t0:.4f} s")
print(f"fib_cache(30): {t3-t2:.6f} s")
print("cache_info de @cache:", fib_cache.cache_info())

### 6.2 `lru_cache` y `maxsize`

`lru_cache` (Least Recently Used) limita el tamaño de la caché para no quedarse sin memoria. Con `maxsize` limitado, los elementos menos usados se expulsan. Útil cuando las entradas son muchas y variadas.

In [ ]:
from functools import lru_cache

@lru_cache(maxsize=3)
def costo(ciudad: str) -> str:
    print("   -> calculando", ciudad)
    return f"costo de {ciudad}"

for c in ["Madrid", "París", "Roma", "Madrid", "Berlín", "Madrid", "París"]:
    print(c, "=>", costo(c))

print("\nmaxsize:", costo.cache_info().maxsize)
print("cache_info:", costo.cache_info())
print("¿Madrid en caché?", costo.cache_info().currsize)

### 6.3 `clear_cache` y `cache_clear`

A veces la caché queda obsoleta (ej.: datos que cambian). Podemos vaciarla bajo demanda con `cache_clear()`.

In [ ]:
from functools import cache

@cache
def impuesto(base: float) -> float:
    print("   -> recalculando impuesto")
    return base * 0.21

print(impuesto(100))   # calcula
print(impuesto(100))   # desde caché (no imprime)
impuesto.cache_clear()
print("Caché vacía tras clear:", impuesto.cache_info())
print(impuesto(100))   # recalcula de nuevo

> **Diferencia:** `@cache` es azúcar de `@lru_cache(maxsize=None)` (sin límite). `@cache` con objetos mutables puede dar resultados obsoletos; usa `lru_cache` cuando el número de entradas sea enorme.

## 7. `@property`, `@classmethod`, `@staticmethod` como decoradores

Estas herramientas que ya conoces **son auténticos decoradores integrados**. Los usaste sin saber que estaban haciendo metaprogramación. Aquí veremos cómo se comportan por dentro.

In [ ]:
class Circulo:
    def __init__(self, radio: float):
        self.radio = radio

    @property
    def area(self) -> float:
        """Área calculada dinámicamente (decorador @property)."""
        import math
        return math.pi * self.radio ** 2

    @classmethod
    def desde_diametro(cls, d: float) -> "Circulo":
        """Constructor alternativo (recibe la clase cls)."""
        return cls(d / 2)

    @staticmethod
    def es_valido(r: float) -> bool:
        """Validación sin necesidad de instancia ni clase."""
        return r > 0

print("@property accede como atributo:", Circulo(2).area)
print("@classmethod:", Circulo.desde_diametro(4).radio)
print("@staticmethod:", Circulo.es_valido(-1))

### 7.1 Implementá lo que hace `@staticmethod` por dentro

`staticmethod` recibe la función y la devuelve **tal cual** (sin vínculo con la instancia), para que al llamarla vía la clase no se pase `self`. Veamos un mini-mimic simple:

In [ ]:
class MiEstatico:
    def __init__(self, fn):
        self.fn = fn

    def __get__(self, instance, owner):
        # Descriptor: devuelve la función pura, sin ligar la instancia
        return self.fn

class Demo:
    @MiEstatico
    def hola():
        return "hola desde estático"

print(Demo.hola())
print(Demo().hola())  # no recibe self

### 7.2 `@property` por dentro

`property` es un **descriptor** que intercepta el acceso al atributo y ejecuta el getter. La sintaxis `@property` sobre el getter y `@area.setter` (decoradores con método) forman toda la fachada.

In [ ]:
class Termometro:
    def __init__(self):
        self._celsius = 0

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, valor):
        if valor < -273.15:
            raise ValueError("Temperatura por debajo del cero absoluto")
        self._celsius = valor

    @property
    def fahrenheit(self):
        return self._celsius * 9 / 5 + 32

t = Termometro()
t.celsius = 25
print("Celsius:", t.celsius, "| Fahrenheit:", t.fahrenheit)
try:
    t.celsius = -300
except ValueError as e:
    print("Validación OK:", e)

## 8. Decoradores anidados (stack)

Puedes apilar varios decoradores. **Se aplican de abajo hacia arriba** (el más cercano a la función se aplica primero) y se *desenvuelven* de arriba hacia abajo al llamar. Es como apilar envolturas de regalo: la última capa que pones es la primera que se quita.

In [ ]:
import functools

def subrayar(fn):
    @functools.wraps(fn)
    def e(*a, **k):
        return f"__{fn(*a, **k)}__"
    return e

def negrita(fn):
    @functools.wraps(fn)
    def e(*a, **k):
        return f"**{fn(*a, **k)}**"
    return e

def italica(fn):
    @functools.wraps(fn)
    def e(*a, **k):
        return f"*{fn(*a, **k)}*"
    return e

@subrayar
@negrita
@italica
def texto():
    return "Python"

# Orden de aplicación (de abajo hacia arriba):
# texto = subrayar( negrita( italica(texto) ) )
print(texto())  # __**  *Python*  **__
print("Nombre:", texto.__name__)

### 8.1 Verificar el orden de aplicación

Imprimamos pasos intermedios para confirmar que los decoradores se aplican de abajo hacia arriba.

In [ ]:
import functools

def etiqueta(nombre):
    def deco(fn):
        @functools.wraps(fn)
        def e(*a, **k):
            print(f"entrando en {nombre} envoltura")
            r = fn(*a, **k)
            print(f"saliendo de {nombre} envoltura")
            return r
        return e
    return deco

@etiqueta("A")
@etiqueta("B")
def operacion():
    print("  [operación central]")
    return "ok"

print("Resultado:", operacion())
print("\n-- Se aplican B primero, luego A; al llamar, entra A y luego B.")

## Diagrama de flujo

Así es como un decorador envuelve (y por tanto *prioriza* el orden) una función:

```
                      APLICACIÓN (de abajo hacia arriba)
  ------------------------------------------------------------------>

  f = decorador1( decorador2( decorador3( f_original ) ) )

  ------------------------------------------------------------------>

              LLAMADA (se desenvuelve de arriba hacia abajo)

              +---------------------------------------------------+
              |  decorador1 (capa más externa)                     |
              |   +---------------------------------------------+ |
              |   |  decorador2                                 | |
              |   |   +---------------------------------------+ | |
              |   |   |  decorador3                           | | |
              |   |   |   +---------------------------------+ | | |
              |   |   |   |   FUNCIÓN ORIGINAL              | | | |
              |   |   |   +---------------------------------+ | | |
              |   |   +---------------------------------------+ | |
              |   +---------------------------------------------+ |
              +---------------------------------------------------+
```

## Tabla de referencia: decoradores de la stdlib

| Decorador | Módulo | Qué hace | Tipo de objetivo | Ejemplo de uso |
|-----------|--------|----------|------------------|----------------|
| `@lru_cache(maxsize=128)` | `functools` | Memoiza con límite LRU | funciones | funciones costosas con entradas repetidas |
| `@cache` | `functools` | Memoiza sin límite (`maxsize=None`) | funciones | recursión como `fib` |
| `@wraps` | `functools` | Copia metadata de la original a la envoltura | envolturas | SIEMPRE en decoradores propios |
| `@property` | `builtins` | Expone método como atributo de solo lectura | métodos | getters calculados |
| `@x.setter` / `@x.deleter` | `builtins` | Define setter/deleter del property | métodos | atributos con validación |
| `@classmethod` | `builtins` | Método ligado a la clase (recibe `cls`) | métodos | constructores alternativos |
| `@staticmethod` | `builtins` | Método sin `self` ni `cls` | métodos | funciones utilitarias de la clase |
| `@dataclass` | `dataclasses` | Genera `__init__`, `__repr__`, `__eq__`... | clases | modelos de datos |
| `@total_ordering` | `functools` | Completa los comparadores faltantes | clases | clases con `__lt__` y quieres el resto |
| `@singledispatch` | `functools` | Dispatch por tipo del primer argumento | funciones | sobrecarga por tipo |
| `@cached_property` | `functools` | Property memoizada a nivel de instancia | métodos | cálculos caros solo una vez por instancia |
| `@abstractmethod` | `abc` | Marca métodos abstractos | métodos | clases base abstractas |

## Ejercicios

### Ejercicio 1 (guiado): Closure que recuerda

Crea una función `crear_acumulador()` que devuelva una closure que va **sumando** los valores que se le pasan, recordando el total acumulado y devolviéndolo. Pista: usa `nonlocal`.

```python
acc = crear_acumulador()
acc(5)   # -> 5
acc(3)   # -> 8
acc(10)  # -> 18
```

In [ ]:
def crear_acumulador():
    total = 0
    def acumular(valor):
        nonlocal total
        total += valor
        return total
    return acumular

acc = crear_acumulador()
print(acc(5))   # 5
print(acc(3))   # 8
print(acc(10))  # 18

### Ejercicio 2 (guiado): Decorador con `wraps`

Escribe un decorador `@log` que **imprima** el nombre de la función y sus argumentos cada vez que se llame, y que **preserve** `__name__` y `__doc__` usando `functools.wraps`. Pruébalo con una función que sume dos números.

In [ ]:
import functools

def log(fn):
    @functools.wraps(fn)
    def envoltura(*args, **kwargs):
        print(f"llamando {fn.__name__} con args={args}, kwargs={kwargs}")
        return fn(*args, **kwargs)
    return envoltura

@log
def sumar(a, b):
    """Suma dos números."""
    return a + b

print("Resultado:", sumar(3, 4))
print("Nombre:", sumar.__name__)
print("Doc:", sumar.__doc__)

### Ejercicio 3 (guiado): Memoización inversa

Usa `@lru_cache` para acelerar el cálculo de números **triangulares** recursivos: `T(0)=0`, `T(n) = n + T(n-1)`. Calcula `T(1000)` y muestra `cache_info()`.

In [ ]:
from functools import lru_cache

@lru_cache(maxsize=None)
def triangular(n):
    if n == 0:
        return 0
    return n + triangular(n - 1)

print("T(1000) =", triangular(1000))
print(triangular.cache_info())
print("Verificación 1000*1001/2 =", 1000 * 1001 // 2)

### Ejercicio 4 (independiente): Decorador de reintento (**retry**)

Crea un decorador con argumentos `@reintentar(intentos=3, retraso=0.1)` que ejecute la función; si lanza una excepción, espera `retraso` segundos y vuelve a intentarlo hasta `intentos` veces. Si se agotan los intentos, relanza la última excepción. Pista: usa el patrón de **triple función** del tema 4 y mide con `time.sleep`.

Prueba con una función que falle las dos primeras llamadas y tenga éxito a la tercera.

In [ ]:
import functools
import time

def reintentar(intentos: int = 3, retraso: float = 0.1):
    def decorador(fn):
        @functools.wraps(fn)
        def envoltura(*args, **kwargs):
            ultimo_error = None
            for i in range(1, intentos + 1):
                try:
                    return fn(*args, **kwargs)
                except Exception as e:
                    ultimo_error = e
                    print(f"  intento {i} falló: {e!r}; reintentando en {retraso}s")
                    time.sleep(retraso)
            raise ultimo_error
        return envoltura
    return decorador

@reintentar(intentos=3, retraso=0.05)
def tarea_inestable(estado):
    if estado < 2:   # falla las dos primeras veces...
        raise ConnectionError(f"fallo temporal (estado={estado})")
    return "éxito alcanzado"

# Simulamos que la primera llamada falla dos veces y tiene éxito en la tercera
import itertools
contador = itertools.count(0)

@reintentar(intentos=3, retraso=0.05)
def tarea():
    n = next(contador)
    if n < 2:
        raise ConnectionError(f"estado={n}")
    return "éxito"

print("Resultado final:", tarea())

## Resumen

- **Primera clase:** las funciones se asignan, pasan y retornan como cualquier objeto.
- **Closure:** función interna que captura el *enclosing scope* (`__closure__`); `nonlocal` permite modificarlo.
- **Decorador:** `@deco` es `f = deco(f)`. Envuelve sin modificar el original.
- **`functools.wraps`:** imprescindible para conservar `__name__`/`__doc__` en toda envoltura.
- **Con argumentos:** patrón de **triple función** (factory → decorador → envoltura).
- **Clases:** también se decoran (`@dataclass`, registro, añadir métodos).
- **Memoización:** `@cache` y `@lru_cache` reducen cálculos recursivos de exponencial a lineal.
- **Stack:** los decoradores se aplican **de abajo hacia arriba** y se desenvuelven al revés.
- **stdlib:** `@property`, `@classmethod`, `@staticmethod`... son decoradores con descriptores por dentro.

> Los decoradores y closures te permiten escribir código **DRY**, elegante y reutilizable: añadir timing, retry, caché o validación con una sola línea, sin invadir la lógica de negocio.